# Tutorial 110 — Combine retrieval channels

Semantic, lexical, and code-aware retrieval rank the same repository using
different signals and incomparable raw score scales. This tutorial combines
their rank positions with Reciprocal Rank Fusion (RRF).

You will run the three channels separately, inspect how their rankings differ,
apply RRF, and trace the winning hybrid results back to their positions in each
original list. RRF rewards evidence that several independent channels rank
highly without requiring their raw scores to be calibrated against one another.

[Tutorial 120](./120_retrieval_routing.ipynb) keeps this ranking unchanged and
adds deterministic context when a question explicitly names a repository file,
function, or code relationship.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import qachat


## 2. Load an embedding model

Only the semantic channel requires an embedding model. The backend is resolved
first and the matching canonical QAChat embedding-model default is selected.
The lexical and code-aware channels remain deterministic and model-independent.


In [ ]:
REQUESTED_BACKEND = qachat.BACKEND_AUTO

SELECTED_BACKEND = qachat.choose_backend(REQUESTED_BACKEND)

MODEL_ID = (
    qachat.DEFAULT_EMBEDDING_MODEL_MLX
    if SELECTED_BACKEND == "mlx"
    else qachat.DEFAULT_EMBEDDING_MODEL_TORCH
)

print(f"Requested backend: {REQUESTED_BACKEND}")
print(f"Resolved backend: {SELECTED_BACKEND}")
print(f"Embedding model: {MODEL_ID}")


In [ ]:
ALLOW_DOWNLOAD = True


In [ ]:
embedding_session = qachat.EmbeddingSession(
    MODEL_ID,
    backend=SELECTED_BACKEND,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 3. Prepare repository data

Build one consistent repository representation and derive the three retrieval
structures introduced in Tutorials
[70](./70_semantic_retrieval.ipynb),
[80](./80_lexical_retrieval.ipynb), and
[90](./90_code_aware_retrieval.ipynb). This ensures that the semantic, lexical,
and code-aware rankings refer to compatible chunks and source metadata.


In [ ]:
report = qachat.scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat"],
)


In [ ]:
source_files = qachat.read_repository_files(report)
units = qachat.extract_source_units(source_files)


In [ ]:
chunks = qachat.chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


## 4. Define one question

The question names a real function. This gives all three channels useful but
different evidence: embeddings can match the surrounding meaning, lexical
retrieval can match the identifier and related words, and code-aware retrieval
can follow the function's structural relationships.


In [ ]:
QUESTION = "How does semantic_search rank repository chunks?"


In [ ]:
semantic_query = qachat.format_qwen3_query(
    instruction=qachat.SOFTWARE_REPOSITORY_INSTRUCTION,
    question=QUESTION,
)


## 5. Run each retrieval channel

Run semantic, lexical, and code-aware retrieval separately before combining
anything. Keeping the intermediate rankings visible is important for
understanding what hybrid retrieval contributes.


In [ ]:
semantic_index = qachat.build_semantic_index(
    chunks,
    embedding_session,
)

semantic_results = qachat.semantic_search(
    semantic_index,
    semantic_query,
    embedding_session,
    count=10,
)


In [ ]:
lexical_index = qachat.build_lexical_index(chunks)

lexical_results = qachat.lexical_search(
    lexical_index,
    QUESTION,
    count=10,
)


In [ ]:
code_graph = qachat.build_code_graph(source_files, units)

code_results = qachat.code_search(
    code_graph,
    chunks,
    QUESTION,
    count=10,
)


## 6. Inspect the individual rankings

The three methods rank the same repository differently because they measure
different signals. Their raw scores are not calibrated to one another, so the
next step combines rank positions rather than comparing raw score values.


In [ ]:
def show_ranking(name, results, count=5):
    print(f"\n{name}")
    for rank, result in enumerate(results[:count], start=1):
        label = result.chunk.symbol or result.chunk.path
        print(f"{rank}. {label}")


show_ranking("Semantic", semantic_results)
show_ranking("Lexical", lexical_results)
show_ranking("Code-aware", code_results)


## 7. Fuse the rankings with RRF

Reciprocal Rank Fusion (RRF) assigns a contribution based on a result's rank in
each list. With the conventional offset used here, a result at rank `r`
contributes `1 / (60 + r)` from that channel. Results supported by several
channels accumulate several contributions.

This makes RRF robust to the fact that semantic cosine similarity, lexical
TF-IDF similarity, and code-aware scores live on different numerical scales.


In [ ]:
hybrid_results = qachat.hybrid_search(
    semantic=semantic_results,
    lexical=lexical_results,
    code=code_results,
    count=5,
)

for rank, result in enumerate(hybrid_results, start=1):
    print(
        f"{rank}. {result.score:.4f} "
        f"{result.channels} "
        f"{result.chunk.symbol}"
    )


## 8. Inspect how RRF combined them

Map each chunk back to its position in the three original rankings. This makes
the hybrid score interpretable: a result can become the best combined evidence
without being rank 1 in any single channel if several channels consistently
place it near the top.


In [ ]:
def rank_positions(results):
    return {
        result.chunk.identifier: rank
        for rank, result in enumerate(results, start=1)
    }


semantic_ranks = rank_positions(semantic_results)
lexical_ranks = rank_positions(lexical_results)
code_ranks = rank_positions(code_results)


Now compare where each hybrid result appeared in the three original
rankings.


In [ ]:
print(
    f"{'Chunk':<32}"
    f"{'Semantic':>10}"
    f"{'Lexical':>10}"
    f"{'Code':>10}"
    f"{'RRF':>12}"
)

for result in hybrid_results:
    identifier = result.chunk.identifier
    name = result.chunk.symbol or result.chunk.path.name

    print(
        f"{str(name):<32}"
        f"{str(semantic_ranks.get(identifier, '-')):>10}"
        f"{str(lexical_ranks.get(identifier, '-')):>10}"
        f"{str(code_ranks.get(identifier, '-')):>10}"
        f"{result.score:>12.4f}"
    )


A result does not need to rank first in any single channel to become the
best hybrid result. RRF rewards chunks that several independent retrieval
methods rank highly.


## 9. Inspect the best hybrid evidence

The highest-ranked hybrid result is the chunk RRF considers strongest after
combining the three retrieval channels.


In [ ]:
best = hybrid_results[0]

print(f"Best source: {best.chunk.path}")
print(f"Channels: {best.channels}\n")
print(best.chunk.text)


## 10. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Continue the tutorial series

**Previous:** [Tutorial 100 — Persist retrieval indexes](./100_persistent_indexes.ipynb)

**Next:** [Tutorial 120 — Expand retrieval deterministically](./120_retrieval_routing.ipynb)
